In [1]:
# import library
import os
import re
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from neo4j import GraphDatabase

In [2]:
# load config dari .env
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL_NAME = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b:free")

NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
print("MODEL:", MODEL_NAME)
print("NEO4J_URI:", NEO4J_URI)

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY belum diisi di file .env")
if not NEO4J_PASSWORD:
    raise ValueError("NEO4J_PASSWORD belum diisi di file .env")

MODEL: openai/gpt-oss-120b:free
NEO4J_URI: neo4j://127.0.0.1:7687


In [3]:
# koneksi openrouter dan neo4j
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)
driver.verify_connectivity()
print("Neo4j connection successful.")

Neo4j connection successful.


In [4]:
# load dataset deskripsi species
csv_path = "C:/Users/Rendra/Downloads/species_descriptions.csv"

df = pd.read_csv(csv_path)
print(df.shape)
print(df.columns)
df.head()

(50, 2)
Index(['species', 'text'], dtype='object')


,species,text
0,Poltys illepidus,Poltys illepidus is a species from the genus P...
1,Centella asiatica,Centella asiatica is a species from the genus ...
2,Euphorbia hirta,Euphorbia hirta is a species from the genus Eu...
3,Bryobium retusum,Bryobium retusum is a species from the genus B...
4,Crepidium koordersii,Crepidium koordersii is a species from the gen...


In [5]:
# deteksi kolom species dan deskripsi 
possible_species_cols = ["species", "scientificName", "scientific_name", "name", "species_name"]
possible_text_cols = ["description", "text", "species_description", "document", "content"]
species_col = None
text_col = None

for col in possible_species_cols:
    if col in df.columns:
        species_col = col
        break
for col in possible_text_cols:
    if col in df.columns:
        text_col = col
        break

print("Species column:", species_col)
print("Text column:", text_col)

if species_col is None:
    raise ValueError("Kolom nama species tidak ditemukan. Cek nama kolom CSV kamu.")
if text_col is None:
    raise ValueError("Kolom deskripsi tidak ditemukan. Cek nama kolom CSV kamu.")

Species column: species
Text column: text


In [6]:
# prompt graph builder 
GRAPH_BUILDER_PROMPT = """
You are an information extraction assistant for a biodiversity knowledge graph.
Your task is to extract structured graph information from a species description.
Extract only the following fields:
1. habitats
2. threats
3. environments

Return ONLY valid JSON with this exact format:
{
  "habitats": [],
  "threats": [],
  "environments": []
}
Rules:
- Do not add explanation.
- Do not use markdown.
- Use concise lowercase phrases.
- If information is not available, return an empty list.
- habitats should describe places where the species lives, such as forest, river, wetland, grassland, coral reef, coastal area.
- threats should describe risks or disturbances, such as habitat loss, pollution, climate change, hunting, invasive species.
- environments should describe broader ecological settings, such as tropical forest, freshwater, marine, urban, agricultural land.
"""

In [7]:
# fungsi cleaning json dari llm
def clean_json_output(text):
    text = text.strip()
    text = re.sub(r"```json", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1:
        text = text[start:end + 1]
    return text

def extract_graph_info(description):
    messages = [
        {
            "role": "system",
            "content": GRAPH_BUILDER_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Species description:
{description}
Extract graph information as JSON:
"""
        }
    ]
    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0,
        max_tokens=500
    )
    raw_output = response.choices[0].message.content
    cleaned_output = clean_json_output(raw_output)
    try:
        data = json.loads(cleaned_output)
    except json.JSONDecodeError:
        print("JSON parse error. Raw output:")
        print(raw_output)
        data = {
            "habitats": [],
            "threats": [],
            "environments": []
        }
    for key in ["habitats", "threats", "environments"]:
        if key not in data or not isinstance(data[key], list):
            data[key] = []
    return data

In [8]:
# test ekstraksi 1 deskripsi
sample_row = df.iloc[0]
sample_species = sample_row[species_col]
sample_description = sample_row[text_col]
print("SPECIES:", sample_species)
print("DESCRIPTION:")
print(sample_description)

sample_extraction = extract_graph_info(sample_description)
print("\nEXTRACTION RESULT:")
print(json.dumps(sample_extraction, indent=2))

SPECIES: Poltys illepidus
DESCRIPTION:
Poltys illepidus is a species from the genus Poltys and family Araneidae. It belongs to the order Araneae, class Arachnida, and kingdom Animalia. This species has occurrence records in Indonesia, which is part of Southeast Asia. It is commonly associated with habitats such as natural habitats, forests, wetlands, and human-modified environments. Potential threats to this species include habitat degradation and environmental change.

EXTRACTION RESULT:
{
  "habitats": [
    "natural habitat",
    "forest",
    "wetland",
    "human-modified environment"
  ],
  "threats": [
    "habitat degradation",
    "environmental change"
  ],
  "environments": [
    "tropical forest",
    "wetland",
    "human-modified"
  ]
}


In [9]:
# coba 5 data 
MAX_ROWS = 5
extracted_rows = []
for idx, row in df.head(MAX_ROWS).iterrows():
    species_name = row[species_col]
    description = row[text_col]
    extracted = extract_graph_info(description)
    extracted_rows.append({
        "species": species_name,
        "description": description,
        "habitats": extracted["habitats"],
        "threats": extracted["threats"],
        "environments": extracted["environments"]
    })
extracted_df = pd.DataFrame(extracted_rows)
extracted_df

,species,description,habitats,threats,environments
0,Poltys illepidus,Poltys illepidus is a species from the genus P...,"[natural habitat, forest, wetland, human-modif...","[habitat degradation, environmental change]","[tropical forest, wetland, human-modified]"
1,Centella asiatica,Centella asiatica is a species from the genus ...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land, open area]"
2,Euphorbia hirta,Euphorbia hirta is a species from the genus Eu...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land, disturbed..."
3,Bryobium retusum,Bryobium retusum is a species from the genus B...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land, disturbed..."
4,Crepidium koordersii,Crepidium koordersii is a species from the gen...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land, open area]"


In [10]:
# proses 15 data
MAX_ROWS = 15
extracted_rows = []
for idx, row in df.head(MAX_ROWS).iterrows():
    species_name = row[species_col]
    description = row[text_col]
    print(f"Processing {idx + 1}/{MAX_ROWS}: {species_name}")
    try:
        extracted = extract_graph_info(description)
        extracted_rows.append({
            "species": species_name,
            "description": description,
            "habitats": extracted["habitats"],
            "threats": extracted["threats"],
            "environments": extracted["environments"]
        })
    except Exception as e:
        print("Error saat memproses:", species_name)
        print(e)
        print("Proses dihentikan. Data yang sudah berhasil tetap disimpan.")
        break
extracted_df = pd.DataFrame(extracted_rows)
extracted_df

Processing 1/15: Poltys illepidus
Processing 2/15: Centella asiatica
Processing 3/15: Euphorbia hirta
Processing 4/15: Bryobium retusum
Processing 5/15: Crepidium koordersii
Processing 6/15: Anoplolepis gracilipes
Processing 7/15: Euphorbia heterophylla
Processing 8/15: Tapinoma melanocephalum
Processing 9/15: Doleschallia bisaltide
Processing 10/15: Breynia androgyna
Processing 11/15: Suastus gremius
Processing 12/15: Ceratosoma tenue
Processing 13/15: Catopsilia pomona
Processing 14/15: Phyllodesmium briareum
Processing 15/15: Scytodes fusca


,species,description,habitats,threats,environments
0,Poltys illepidus,Poltys illepidus is a species from the genus P...,"[natural habitat, forest, wetland, human-modif...","[habitat degradation, environmental change]","[tropical forest, wetland, human-modified]"
1,Centella asiatica,Centella asiatica is a species from the genus ...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land, disturbed..."
2,Euphorbia hirta,Euphorbia hirta is a species from the genus Eu...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land, open area]"
3,Bryobium retusum,Bryobium retusum is a species from the genus B...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land, open area]"
4,Crepidium koordersii,Crepidium koordersii is a species from the gen...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land]"
5,Anoplolepis gracilipes,Anoplolepis gracilipes is a species from the g...,"[tropical forest, vegetation area, garden, fre...","[habitat degradation, pesticide exposure, envi...","[tropical forest, garden, freshwater, vegetati..."
6,Euphorbia heterophylla,Euphorbia heterophylla is a species from the g...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land, open area]"
7,Tapinoma melanocephalum,Tapinoma melanocephalum is a species from the ...,"[tropical forest, vegetation area, garden, fre...","[habitat degradation, pesticide exposure, envi...","[tropical forest, freshwater, anthropogenic ga..."
8,Doleschallia bisaltide,Doleschallia bisaltide is a species from the g...,"[tropical forest, vegetation area, garden, fre...","[habitat degradation, pesticide exposure, envi...","[tropical forest, freshwater, garden]"
9,Breynia androgyna,Breynia androgyna is a species from the genus ...,"[tropical forest, open area, disturbed habitat...","[land conversion, habitat degradation, environ...","[tropical forest, agricultural land, disturbed..."


In [11]:
# save hasil ekstraksi ke csv
output_path = "C:/Users/Rendra/Downloads/llm_graph_builder_extraction.csv"
extracted_df.to_csv(output_path, index=False)
print("Saved to:", output_path)

Saved to: C:/Users/Rendra/Downloads/llm_graph_builder_extraction.csv


In [12]:
# fungsi import hasil graph builder ke neo4j
def run_write_query(query, parameters=None):
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(query, parameters or {})
def import_graph_builder_result(species, habitats, threats, environments):
    species = str(species).strip()
    for habitat in habitats:
        habitat = str(habitat).strip().lower()
        if habitat:
            run_write_query(
                """
                MERGE (s:Species {name: $species})
                MERGE (h:Habitat {name: $habitat})
                MERGE (s)-[:HAS_HABITAT]->(h)
                """,
                {"species": species, "habitat": habitat}
            )
    for threat in threats:
        threat = str(threat).strip().lower()
        if threat:
            run_write_query(
                """
                MERGE (s:Species {name: $species})
                MERGE (t:Threat {name: $threat})
                MERGE (s)-[:HAS_THREAT]->(t)
                """,
                {"species": species, "threat": threat}
            )
    for environment in environments:
        environment = str(environment).strip().lower()
        if environment:
            run_write_query(
                """
                MERGE (s:Species {name: $species})
                MERGE (e:Environment {name: $environment})
                MERGE (s)-[:FOUND_IN_ENVIRONMENT]->(e)
                """,
                {"species": species, "environment": environment}
            )

In [13]:
# import ke neo4j
for idx, row in extracted_df.iterrows():
    import_graph_builder_result(
        species=row["species"],
        habitats=row["habitats"],
        threats=row["threats"],
        environments=row["environments"]
    )
print("LLM Graph Builder results imported to Neo4j.")

LLM Graph Builder results imported to Neo4j.


In [14]:
# cek jumlah node hasil graph builder 
def run_cypher(query):
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query)
        records = [record.data() for record in result]
    return pd.DataFrame(records)
graph_builder_node_count = run_cypher("""
MATCH (n)
WHERE n:Habitat OR n:Threat OR n:Environment
RETURN labels(n)[0] AS label, count(n) AS total
ORDER BY total DESC
""")
graph_builder_node_count

,label,total
0,Environment,13
1,Habitat,11
2,Threat,4


In [15]:
# cek relationship hasil graph builder
graph_builder_rel_count = run_cypher("""
MATCH (:Species)-[r:HAS_HABITAT|HAS_THREAT|FOUND_IN_ENVIRONMENT]->()
RETURN type(r) AS relationship, count(r) AS total
ORDER BY total DESC
""")
graph_builder_rel_count

,relationship,total
0,HAS_HABITAT,60
1,FOUND_IN_ENVIRONMENT,43
2,HAS_THREAT,41


In [16]:
# preview hasil graph builder 
graph_builder_preview = run_cypher("""
MATCH (s:Species)-[r:HAS_HABITAT|HAS_THREAT|FOUND_IN_ENVIRONMENT]->(n)
RETURN
  s.name AS species,
  type(r) AS relationship,
  labels(n)[0] AS target_label,
  n.name AS target_name
ORDER BY species, relationship
LIMIT 30
""")
graph_builder_preview

,species,relationship,target_label,target_name
0,Anoplolepis gracilipes,FOUND_IN_ENVIRONMENT,Environment,garden
1,Anoplolepis gracilipes,FOUND_IN_ENVIRONMENT,Environment,tropical forest
2,Anoplolepis gracilipes,FOUND_IN_ENVIRONMENT,Environment,freshwater
3,Anoplolepis gracilipes,FOUND_IN_ENVIRONMENT,Environment,vegetation area
4,Anoplolepis gracilipes,HAS_HABITAT,Habitat,garden
5,Anoplolepis gracilipes,HAS_HABITAT,Habitat,vegetation area
6,Anoplolepis gracilipes,HAS_HABITAT,Habitat,tropical forest
7,Anoplolepis gracilipes,HAS_HABITAT,Habitat,freshwater surroundings
8,Anoplolepis gracilipes,HAS_THREAT,Threat,habitat degradation
9,Anoplolepis gracilipes,HAS_THREAT,Threat,pesticide exposure


In [17]:
# tutup koneksi
driver.close()
print("Neo4j connection closed.")

Neo4j connection closed.
